# Recipe 15: Find Overlapping Observations

Find products from different instruments that observed the same target at the same time.

## What You'll Learn
- Temporal filtering for specific time windows
- Grouping results by instrument
- Finding coordinated observations

In [ ]:
import pds.peppi as pep
from datetime import datetime, timedelta

In [ ]:
client = pep.PDSRegistryClient()

# Define a specific time window
target_date = datetime(2020, 6, 15)
window = timedelta(days=1)

# Search for Mars observations in this window
# .fields() limits the payload to only what we need, keeping the query fast
products = pep.Products(client) \
    .has_target("Mars") \
    .after(target_date - window) \
    .before(target_date + window) \
    .fields(["ref_lid_instrument", "pds:Time_Coordinates.pds:start_date_time"]) \
    .observationals()

# Group by instrument (cap at 500 products to keep the demo fast)
instruments = {}
MAX_PRODUCTS = 500

for i, product in enumerate(products):
    if i >= MAX_PRODUCTS:
        break
    inst = product.properties.get('ref_lid_instrument', ['Unknown'])[0]
    start = product.properties.get('pds:Time_Coordinates.pds:start_date_time', ['N/A'])[0]
    
    if inst not in instruments:
        instruments[inst] = []
    
    instruments[inst].append({
        'id': product.id,
        'start': start
    })

# Display results
print(f"Observations of Mars around {target_date.date()}:")
for inst, obs_list in instruments.items():
    print(f"\n{inst}: {len(obs_list)} observations")
    for obs in obs_list[:3]:  # Show first 3
        print(f"  {obs['start']}")

## Key Takeaways

- Temporal windows help find coordinated observations
- Grouping by instrument reveals multi-instrument campaigns
- Can identify opportunities for cross-instrument analysis